In [18]:
import os
from typing import Annotated,TypedDict
from langchain.messages import SystemMessage, HumanMessage, AIMessage,RemoveMessage
from langchain_core.messages import BaseMessage
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph,message,MessagesState,START,END




In [19]:
model=ChatGroq(model="qwen/qwen3-32b",temperature=0.2,groq_api_key=os.getenv('GROQ_API_KEY'))

In [20]:
def call_model(state:MessagesState):
    messages=state['messages']
    response=model.invoke(messages)
    return {'messages': [response]}

In [25]:
def delete_msg(state:MessagesState):
    msgs=state['messages']
    if len(msgs)>10:
        remove_msg=msgs[:6]
        return {'messages': [RemoveMessage(id=m.id) for m in remove_msg]}
    
        
    return {}

In [26]:
builder=StateGraph(MessagesState)
builder.add_node("call_model",call_model)
builder.add_node("delete_msg",delete_msg)
builder.add_edge(START,"call_model")
builder.add_edge("call_model","delete_msg")
builder.add_edge("delete_msg",END)

In [27]:
checkpointer=InMemorySaver()
graph=builder.compile(checkpointer=checkpointer)

In [28]:
t1={"configurable":{"thread_id":"thread1"}}


graph.invoke({"messages":[HumanMessage(content="hello my name is Rahul")]},t1)
graph.invoke({"messages":[HumanMessage(content="what is langgraph")]},t1)
graph.invoke({"messages":[HumanMessage(content="what is langchain")]},t1)
graph.invoke({"messages":[HumanMessage(content="hi what is your name ")]},t1)
graph.invoke({"messages":[HumanMessage(content="what is deep learning")]},t1)
graph.invoke({"messages":[HumanMessage(content="who is modi")]},t1)
graph.invoke({"messages":[HumanMessage(content="go to hell translate it into hindi ")]},t1)

{'messages': [HumanMessage(content='hi what is your name ', additional_kwargs={}, response_metadata={}, id='7daa992d-be79-4995-946a-aefcbabfc5cc'),
  AIMessage(content='<think>\nOkay, the user just asked, "hi what is your name." Let me think about how to respond.\n\nFirst, I need to remember the previous conversation. The user introduced himself as Rahul and asked about LangGraph and LangChain. I provided detailed explanations. Now he\'s asking for my name. \n\nI should respond in a friendly and helpful manner. My name is Qwen, so I\'ll mention that. Also, since he\'s been asking about LangChain and LangGraph, maybe I can offer further assistance. \n\nI should keep the tone positive and open-ended. Maybe add an emoji to keep it friendly. Let me check if there\'s anything else he might need. Since he\'s new to these frameworks, offering help with examples or specific questions could be useful.\n\nMake sure the response is concise but not too short. Avoid technical jargon here since it\'

In [29]:
snap = graph.get_state(config=t1)
print("Stored messages after cleanup:", len(snap.values["messages"]))

Stored messages after cleanup: 8


In [30]:
sanps=graph.get_state(config=t1)
valus=sanps.values
for v in valus.get("messages",[]):
    print("-",type(v).__name__,":",v.content)

- HumanMessage : hi what is your name 
- AIMessage : <think>
Okay, the user just asked, "hi what is your name." Let me think about how to respond.

First, I need to remember the previous conversation. The user introduced himself as Rahul and asked about LangGraph and LangChain. I provided detailed explanations. Now he's asking for my name. 

I should respond in a friendly and helpful manner. My name is Qwen, so I'll mention that. Also, since he's been asking about LangChain and LangGraph, maybe I can offer further assistance. 

I should keep the tone positive and open-ended. Maybe add an emoji to keep it friendly. Let me check if there's anything else he might need. Since he's new to these frameworks, offering help with examples or specific questions could be useful.

Make sure the response is concise but not too short. Avoid technical jargon here since it's just a name question. Alright, time to put it all together.
</think>

Hello Rahul! 😊  
My name is **Qwen** (also known as **通义千问*